# 02 - Data Cleaning

Goal: turn the raw transaction data into a clean, analysis-ready dataset.

Cleaning steps, in order:
1. Remove cancellation invoices (Invoice starts with "C")
2. Remove non-positive quantities (returns/adjustments already gone with cancellations, but some negative qty exists outside "C" invoices too)
3. Remove non-positive prices (includes the "Adjust bad debt" accounting rows, which aren't real product sales)
4. Drop exact duplicate rows
5. Compute TotalAmount = Quantity * Price

Order matters here: duplicates are dropped *after* the other filters, not before,
so a duplicate that only exists because of a since-removed cancellation row doesn't
distort the duplicate count.

In [1]:
import pandas as pd

df_2009_2010 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df_2010_2011 = pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

clean_df = df.copy()
clean_df.shape

(1067371, 8)

In [2]:
clean_df["Invoice"] = clean_df["Invoice"].astype(str)

## Remove cancellation invoices

In [3]:
before = len(clean_df)
clean_df = clean_df[~clean_df["Invoice"].str.startswith("C")]
print(f"Removed {before - len(clean_df)} rows. Remaining: {len(clean_df)}")

Removed 19494 rows. Remaining: 1047877


## Remove invalid quantities

In [4]:
before = len(clean_df)
clean_df = clean_df[clean_df["Quantity"] > 0]
print(f"Removed {before - len(clean_df)} rows. Remaining: {len(clean_df)}")

Removed 3457 rows. Remaining: 1044420


Note: this removed 3,457 rows here, on top of the 19,494 cancellation invoices
already gone. So not every negative-quantity row was a "C" invoice — some invalid
quantities show up under normal invoice numbers too. Worth knowing rather than
assuming cancellations and negative quantities were the same set.

## Remove invalid prices

In [5]:
before = len(clean_df)
clean_df = clean_df[clean_df["Price"] > 0]
print(f"Removed {before - len(clean_df)} rows. Remaining: {len(clean_df)}")

Removed 2750 rows. Remaining: 1041670


This removes both the zero-price rows and the 5 negative-price rows
(the "Adjust bad debt" accounting entries) — neither represents a real
product sale.

## Drop duplicate rows

In [6]:
before = len(clean_df)
clean_df = clean_df.drop_duplicates()
print(f"Removed {before - len(clean_df)} rows. Remaining: {len(clean_df)}")

Removed 33757 rows. Remaining: 1007913


A "duplicate" here means every column matches exactly, including InvoiceDate
down to the second and the Invoice number itself. That's not two genuinely
different purchases coinciding — it's the same line item logged twice, so
dropping these is safe and doesn't discard real transactions.

## Compute TotalAmount

In [7]:
clean_df["TotalAmount"] = clean_df["Quantity"] * clean_df["Price"]
clean_df.shape

(1007913, 9)

In [8]:
clean_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalAmount
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


Final shape should be **(1,007,913, 9)** — matches the verified target.

In [9]:
clean_df["Customer ID"].isnull().sum()

np.int64(228488)

228,488 rows still have no Customer ID. We deliberately keep them here —
they're still valid for product-level and revenue-level analysis (EDA, association
rules, NLP on descriptions). We only drop them when building customer-level
features, in a separate dataframe below.

In [10]:
customer_df = clean_df.dropna(subset=["Customer ID"]).copy()
customer_df.shape

(779425, 9)

## Save cleaned data

In [11]:
clean_df.to_csv("../data/processed/retail_clean.csv", index=False)
customer_df.to_csv("../data/processed/retail_customer_level.csv", index=False)
print("saved")

saved
